# Build base dataset

This notebook generates the clean base dataset for the benchmark.

It:
- generates candidate examples for all task families
- validates the candidate pool
- selects the final base examples
- converts them into the clean base dataset format
- validates the final dataset
- exports dataset files and summaries

Outputs:
- `data/candidates/candidates.jsonl`
- `data/reviewed/selected_base_examples.jsonl`
- `data/base/base_examples.jsonl`
- `data/base/base_examples.json`
- `data/base/dataset_summary.json`
- `data/base/validation_report.json`

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('C:/code/testing/LLMs_Robustness_Under_Distractions')

## 1) Imports

In [2]:
from src.generation import (
    generate_all_candidates,
    candidate_to_review_row,
    auto_flag_candidate,
    find_exact_duplicate_inputs,
    count_examples_by_task,
    count_examples_by_status,
    count_by_task_and_status,
    save_candidates_to_jsonl,
    load_candidates_from_jsonl,
    select_base_examples_exact,
)

from src.base_dataset import (
    build_base_dataset,
    build_dataset_summary,
    save_jsonl,
    save_json,
)

from src.validation import (
    validate_dataset,
    summarize_instruction_diversity,
)

## 2) Input/output paths

In [3]:
CANDIDATES_OUTPUT_PATH = PROJECT_ROOT / "data" / "candidates" / "candidates.jsonl"
SELECTED_BASE_EXAMPLES_OUTPUT_PATH = PROJECT_ROOT / "data" / "reviewed" / "selected_base_examples.jsonl"

BASE_JSONL_OUTPUT_PATH = PROJECT_ROOT / "data" / "base" / "base_examples.jsonl"
BASE_JSON_OUTPUT_PATH = PROJECT_ROOT / "data" / "base" / "base_examples.json"
DATASET_SUMMARY_OUTPUT_PATH = PROJECT_ROOT / "data" / "base" / "dataset_summary.json"
VALIDATION_REPORT_OUTPUT_PATH = PROJECT_ROOT / "data" / "base" / "validation_report.json"

(
    CANDIDATES_OUTPUT_PATH,
    SELECTED_BASE_EXAMPLES_OUTPUT_PATH,
    BASE_JSONL_OUTPUT_PATH,
    BASE_JSON_OUTPUT_PATH,
)

(WindowsPath('C:/code/testing/LLMs_Robustness_Under_Distractions/data/candidates/candidates.jsonl'),
 WindowsPath('C:/code/testing/LLMs_Robustness_Under_Distractions/data/reviewed/selected_base_examples.jsonl'),
 WindowsPath('C:/code/testing/LLMs_Robustness_Under_Distractions/data/base/base_examples.jsonl'),
 WindowsPath('C:/code/testing/LLMs_Robustness_Under_Distractions/data/base/base_examples.json'))

## 3) Generate candidate examples

This uses the redesigned task templates and instruction paraphrase pools.

In [4]:
candidates = generate_all_candidates()

len(candidates)

250

In [5]:
count_examples_by_task(candidates)

{'single_label_classification': 50,
 'multi_label_classification': 50,
 'information_extraction': 50,
 'rule_based_transformation': 50,
 'extractive_qa': 50}

## 4) Inspect raw candidate examples

In [6]:
for idx in range(3):
    print("=" * 100)
    print(candidates[idx])

CandidateExample(example_id='slc_000', task_name='single_label_classification', template_name='app_experience_note', instruction='Choose one sentiment label only from {positive, negative, neutral}.', input_data={'text': 'The app looked rough, and most features worked, but using it felt annoying.'}, gold_output={'label': 'negative'}, metadata={'label_set': ['positive', 'negative', 'neutral']}, review_status='pending', review_note='')
CandidateExample(example_id='slc_001', task_name='single_label_classification', template_name='app_experience_note', instruction='Classify the sentiment of the text using exactly one label from {positive, negative, neutral}.', input_data={'text': 'The app looked disappointing, and most features worked, but using it felt slow.'}, gold_output={'label': 'negative'}, metadata={'label_set': ['positive', 'negative', 'neutral']}, review_status='pending', review_note='')
CandidateExample(example_id='slc_002', task_name='single_label_classification', template_name='

## 5) Convert a few candidates to review rows

This makes it easier to inspect the generated inputs and outputs.

In [7]:
for idx in range(5):
    print("=" * 120)
    row = candidate_to_review_row(candidates[idx])
    print(json.dumps(row, indent=2, ensure_ascii=False))

{
  "example_id": "slc_000",
  "task_name": "single_label_classification",
  "template_name": "app_experience_note",
  "instruction": "Choose one sentiment label only from {positive, negative, neutral}.",
  "rendered_input": "The app looked rough, and most features worked, but using it felt annoying.",
  "gold_output": "{\"label\": \"negative\"}",
  "review_status": "pending",
  "review_note": ""
}
{
  "example_id": "slc_001",
  "task_name": "single_label_classification",
  "template_name": "app_experience_note",
  "instruction": "Classify the sentiment of the text using exactly one label from {positive, negative, neutral}.",
  "rendered_input": "The app looked disappointing, and most features worked, but using it felt slow.",
  "gold_output": "{\"label\": \"negative\"}",
  "review_status": "pending",
  "review_note": ""
}
{
  "example_id": "slc_002",
  "task_name": "single_label_classification",
  "template_name": "app_experience_note",
  "instruction": "Return exactly one sentiment l

## 6) Auto-flag potentially problematic candidates

In [8]:
flagged_candidates = []

for example in candidates:
    issues = auto_flag_candidate(example)
    if issues:
        flagged_candidates.append({
            "example_id": example.example_id,
            "task_name": example.task_name,
            "template_name": example.template_name,
            "issues": issues,
        })

len(flagged_candidates)

0

In [9]:
for item in flagged_candidates[:10]:
    print(item)

## 7) Check for exact duplicate rendered inputs

In [10]:
duplicate_inputs = find_exact_duplicate_inputs(candidates)
len(duplicate_inputs)

0

In [11]:
if duplicate_inputs:
    for rendered, ids in list(duplicate_inputs.items())[:5]:
        print("=" * 120)
        print("IDs:", ids)
        print(rendered)
else:
    print("No exact duplicate rendered inputs found.")

No exact duplicate rendered inputs found.


## 8) Save candidate pool

This is useful for manual review and reproducibility.

In [12]:
save_candidates_to_jsonl(candidates, str(CANDIDATES_OUTPUT_PATH))
print("Saved:", CANDIDATES_OUTPUT_PATH)

Saved: C:\code\testing\LLMs_Robustness_Under_Distractions\data\candidates\candidates.jsonl


## 9) Approve all generated candidates for now

If you later add a manual review stage, this cell can be replaced by loading reviewed statuses.
For now, we mark everything as approved so the notebook can produce the full base dataset.

In [13]:
for example in candidates:
    example.review_status = "approved"
    example.review_note = ""

In [14]:
count_examples_by_status(candidates), count_by_task_and_status(candidates)

({'approved': 250},
 {'single_label_classification': {'approved': 50},
  'multi_label_classification': {'approved': 50},
  'information_extraction': {'approved': 50},
  'rule_based_transformation': {'approved': 50},
  'extractive_qa': {'approved': 50}})

## 10) Select exactly 50 examples per task

The helper ensures that the final base pool has the expected size.

In [15]:
selected_candidates = select_base_examples_exact(
    candidates,
    n_per_task=50,
)

len(selected_candidates)

250

In [16]:
selected_task_counts = {}
for example in selected_candidates:
    selected_task_counts[example.task_name] = selected_task_counts.get(example.task_name, 0) + 1

selected_task_counts

{'extractive_qa': 50,
 'information_extraction': 50,
 'multi_label_classification': 50,
 'rule_based_transformation': 50,
 'single_label_classification': 50}

## 11) Save selected candidate examples

In [17]:
save_candidates_to_jsonl(selected_candidates, str(SELECTED_BASE_EXAMPLES_OUTPUT_PATH))
print("Saved:", SELECTED_BASE_EXAMPLES_OUTPUT_PATH)

Saved: C:\code\testing\LLMs_Robustness_Under_Distractions\data\reviewed\selected_base_examples.jsonl


## 12) Build final clean base dataset records

In [18]:
base_records = build_base_dataset(selected_candidates)

len(base_records)

250

In [19]:
print(json.dumps(base_records[0], indent=2, ensure_ascii=False))

{
  "example_id": "qa_200",
  "task_name": "extractive_qa",
  "template_name": "deployment_update_passage",
  "instruction": "Answer the question using an exact span from the passage.",
  "input_data": {
    "passage": "The deployment note says BlueRiver Health will roll out the medical software on 2025-10-03 after testing in Vienna. A separate memo about Vertex Retail Tech concerned a different tool.",
    "question": "Which company will roll out the medical software on 2025-10-03?"
  },
  "gold_output": {
    "answer": "BlueRiver Health"
  },
  "metadata": {
    "answer_field": "company"
  }
}


## 13) Build dataset summary

In [20]:
dataset_summary = build_dataset_summary(base_records)
print(json.dumps(dataset_summary, indent=2, ensure_ascii=False))

{
  "total_records": 250,
  "counts_by_task": {
    "extractive_qa": 50,
    "information_extraction": 50,
    "multi_label_classification": 50,
    "rule_based_transformation": 50,
    "single_label_classification": 50
  },
  "counts_by_template": {
    "app_experience_note": 9,
    "appointment_notice_rich": 9,
    "arrival_record_with_multiple_dates": 8,
    "company_news": 7,
    "conference_attendance_rich": 9,
    "delivery_comment_mixed": 9,
    "deployment_update_passage": 8,
    "event_reaction_short_review": 8,
    "event_registration_rich": 8,
    "event_report": 7,
    "hosted_event_passage": 5,
    "institutional_announcement": 6,
    "meeting_announcement_rich": 8,
    "meeting_city_resolution": 8,
    "mixed_caps_checklist__lowercase": 3,
    "mixed_caps_checklist__remove_punctuation": 2,
    "mixed_caps_checklist__remove_words_longer_than_6": 2,
    "mixed_caps_checklist__replace_numbers_with_NUM": 2,
    "mixed_case_statement__lowercase": 2,
    "mixed_case_statement__

## 14) Inspect instruction diversity

Unlike the older version of the pipeline, instruction paraphrase diversity is now expected and desirable.

In [21]:
instruction_diversity = summarize_instruction_diversity(base_records)
print(json.dumps(instruction_diversity, indent=2, ensure_ascii=False))

{
  "extractive_qa": {
    "num_unique_instructions": 8,
    "instructions": [
      "Answer the question using an exact span from the passage.",
      "Answer with the precise text appearing in the passage.",
      "Copy the answer directly from the passage.",
      "Extract the answer directly from the passage without paraphrasing.",
      "Return only the exact answer span from the passage.",
      "Return the exact words from the passage that answer the question.",
      "Use a verbatim span from the passage.",
      "Use the passage wording exactly for the answer."
    ]
  },
  "information_extraction": {
    "num_unique_instructions": 6,
    "instructions": [
      "Extract person, date, and location and return them as structured fields.",
      "Extract the fields person, date, and location from the text.",
      "Find the person, date, and location mentioned in the text.",
      "Identify and output person, date, and location from the text.",
      "Return the person, date, and

## 15) Validate final base dataset

In [22]:
validation_report = validate_dataset(
    base_records,
    expected_total=250,
)

print("is_valid:", validation_report["is_valid"])
print("num_records_with_issues:", validation_report["num_records_with_issues"])
print("dataset_level_issues:", validation_report["dataset_level_issues"])

is_valid: True
num_records_with_issues: 0
dataset_level_issues: []


In [23]:
print(json.dumps(validation_report, indent=2, ensure_ascii=False))

{
  "is_valid": true,
  "total_records": 250,
  "num_records_with_issues": 0,
  "record_issues": {},
  "dataset_level_issues": [],
  "issue_counts": {},
  "instruction_diversity_summary": {
    "extractive_qa": {
      "num_unique_instructions": 8,
      "instructions": [
        "Answer the question using an exact span from the passage.",
        "Answer with the precise text appearing in the passage.",
        "Copy the answer directly from the passage.",
        "Extract the answer directly from the passage without paraphrasing.",
        "Return only the exact answer span from the passage.",
        "Return the exact words from the passage that answer the question.",
        "Use a verbatim span from the passage.",
        "Use the passage wording exactly for the answer."
      ]
    },
    "information_extraction": {
      "num_unique_instructions": 6,
      "instructions": [
        "Extract person, date, and location and return them as structured fields.",
        "Extract the f

## 16) Save final base dataset artifacts

In [24]:
save_jsonl(base_records, str(BASE_JSONL_OUTPUT_PATH))
save_json(base_records, str(BASE_JSON_OUTPUT_PATH))
save_json(dataset_summary, str(DATASET_SUMMARY_OUTPUT_PATH))
save_json(validation_report, str(VALIDATION_REPORT_OUTPUT_PATH))

print("Saved JSONL:", BASE_JSONL_OUTPUT_PATH)
print("Saved JSON :", BASE_JSON_OUTPUT_PATH)
print("Saved summary:", DATASET_SUMMARY_OUTPUT_PATH)
print("Saved validation:", VALIDATION_REPORT_OUTPUT_PATH)

Saved JSONL: C:\code\testing\LLMs_Robustness_Under_Distractions\data\base\base_examples.jsonl
Saved JSON : C:\code\testing\LLMs_Robustness_Under_Distractions\data\base\base_examples.json
Saved summary: C:\code\testing\LLMs_Robustness_Under_Distractions\data\base\dataset_summary.json
Saved validation: C:\code\testing\LLMs_Robustness_Under_Distractions\data\base\validation_report.json


## 17) Final sanity checks

In [25]:
assert len(base_records) == 250, "Expected 250 base records"
assert Path(BASE_JSONL_OUTPUT_PATH).exists(), "Base JSONL output missing"
assert Path(DATASET_SUMMARY_OUTPUT_PATH).exists(), "Dataset summary output missing"
assert Path(VALIDATION_REPORT_OUTPUT_PATH).exists(), "Validation report output missing"

print("All base-dataset sanity checks passed.")

All base-dataset sanity checks passed.


## 18) Debugging: inspect problematic records if validation fails

In [26]:
if not validation_report["is_valid"]:
    print("Validation failed. Showing first few problematic records.\n")
    shown = 0
    for example_id, issues in validation_report["record_issues"].items():
        print("=" * 100)
        print("example_id:", example_id)
        print("issues:", issues)
        matching = [r for r in base_records if r["example_id"] == example_id]
        if matching:
            print(json.dumps(matching[0], indent=2, ensure_ascii=False))
        shown += 1
        if shown >= 3:
            break
else:
    print("Validation passed with no record-level issues.")

Validation passed with no record-level issues.
